## Chapter 6 Assigned Problems

In [1]:
# As always, we start with our favorite standard imports. 

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
%matplotlib inline

import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.metrics import root_mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.linear_model import LassoCV
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA


## Homework 4 Spring 2026
- 6.6.0 (2+2+2+2+2+2= 12 points) Conceptual questions
  - (a) What is the definition of scale equivariant?
  -  (b) Why is it important to standardize the predictors when using ridge regression or the lasso?
  -  (c) What is the difference between ridge regression and the lasso?
  -  (d) What is an advantage of using ridge regression or the lasso over least squares linear regression?
  -  (e) What is the purpose of PCA?
  -  (f) What does the first principle component maximize?
    
- 6.6.9 (a-g) In this exercise, we will predict the number of applications received
using the other variables in the `College` data set.
  


### Grading distribution 

- 6.60 (12 points)
- 6.6.9 (44 points)


## 6.6.0 Conceptual questions
  - (a) What is the definition of scale equivariant?

###ANSWER## A procedure is scale equivariant if multiplying the input by a constant results in the output being multiplied by a predictable constant factor.

-  (b) Why is it important to standardize the predictors when using ridge regression or the lasso?


###ANSWER### It is important to standardize predictors in ridge regression and the lasso because the penalty term depends on the magnitude of the coefficients. If predictors are on different scales, variables with larger scales will be penalized differently than variables with smaller scales, leading to unfair shrinkage. Standardizing ensures that all predictors are penalized equally.

-  (c) What is the difference between ridge regression and the lasso?
 


####ANSWER### Ridge regression uses an $L_2$ penalty and shrinks coefficients toward zero but does not set them exactly to zero. The lasso uses an $L_1$ penalty and can shrink some coefficients exactly to zero, performing variable selection. Ridge reduces variance, while lasso both reduces variance and selects predictors.

 -  (d) What is an advantage of using ridge regression or the lasso over least squares linear regression?
  

###ANSWER### An advantage of ridge regression and the lasso over least squares is that they reduce overfitting by shrinking the coefficient estimates. This leads to lower variance and often better prediction accuracy, especially when predictors are highly correlated or when the number of predictors is large relative to the sample size.

-  (e) What is the purpose of PCA (Principal Component Analysis)?


###ANSWER### The purpose of PCA is to reduce the dimensionality of a dataset by transforming the original variables into a smaller set of uncorrelated variables (principal components) that capture most of the variation in the data.

-  (f) What does the first principle component maximize?

###ANSWER### The first principal component maximizes the variance of the projected data. It is the linear combination of the original variables that captures the greatest possible variance.

## 6.6.9
In this exercise, we will predict the number of applications received
using the other variables in the `College` data set.

In [2]:
## Load the dataset
url = "https://msu-cmse-courses.github.io/CMSE381-S26/_downloads/cc29ec6408d657de88bc7fe6de6b1170/College.csv"
college_df = pd.read_csv(url)
college_df = college_df.set_index('Unnamed: 0')

## One-hot encode the categorical variable
college_df = pd.get_dummies(college_df, drop_first=True)

college_df.head()

,Apps,Accept,Enroll,Top10perc,Top25perc,F.Undergrad,P.Undergrad,Outstate,Room.Board,Books,Personal,PhD,Terminal,S.F.Ratio,perc.alumni,Expend,Grad.Rate,Private_Yes
Unnamed: 0,,,,,,,,,,,,,,,,,,
Abilene Christian University,1660,1232,721,23,52,2885,537,7440,3300,450,2200,70,78,18.1,12,7041,60,True
Adelphi University,2186,1924,512,16,29,2683,1227,12280,6450,750,1500,29,30,12.2,16,10527,56,True
Adrian College,1428,1097,336,22,50,1036,99,11250,3750,400,1165,53,66,12.9,30,8735,54,True
Agnes Scott College,417,349,137,60,89,510,63,12960,5450,450,875,92,97,7.7,37,19016,59,True
Alaska Pacific University,193,146,55,16,44,249,869,7560,4120,800,1500,76,72,11.9,2,10922,15,True


In [3]:
# Convert entire dataframe to float before modeling
college_df = college_df.astype(float)

print(college_df.dtypes)

Apps           float64
Accept         float64
Enroll         float64
Top10perc      float64
Top25perc      float64
F.Undergrad    float64
P.Undergrad    float64
Outstate       float64
Room.Board     float64
Books          float64
Personal       float64
PhD            float64
Terminal       float64
S.F.Ratio      float64
perc.alumni    float64
Expend         float64
Grad.Rate      float64
Private_Yes    float64
dtype: object


(a) (5 points):Split the data in test set and training set (this will only be used for the linear regression, all other models are cross-validated with the whole set).

In [4]:
###ANSWER###
## Split into predictors and target
y = college_df['Apps']
X = college_df.drop('Apps', axis=1)

## Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

In [5]:
print(X_train.dtypes)

Accept         float64
Enroll         float64
Top10perc      float64
Top25perc      float64
F.Undergrad    float64
P.Undergrad    float64
Outstate       float64
Room.Board     float64
Books          float64
Personal       float64
PhD            float64
Terminal       float64
S.F.Ratio      float64
perc.alumni    float64
Expend         float64
Grad.Rate      float64
Private_Yes    float64
dtype: object


(b) (5 points): Fit a linear model using least squares on the training set, and
report the test error obtained.

In [6]:
####ANSWER###
X_train_sm = sm.add_constant(X_train)
X_test_sm  = sm.add_constant(X_test)

ols = sm.OLS(y_train, X_train_sm).fit()
pred = ols.predict(X_test_sm)
rmse = root_mean_squared_error(y_test, pred)
print("Least Squares Test RMSE:", round(rmse, 2))

Least Squares Test RMSE: 1348.33


#### ✅ Question (b): Documenting Your Solution Process (3 points)

Please answer the following clearly and completely:

1. **Prior Knowledge vs. External Resources (1 point)**  
   Indicate which parts of Question (b) you completed using your own prior knowledge, and which parts you completed using external resources (e.g., generative AI, past assignments, Stack Overflow, Google, etc.).

In [7]:
###YOUR ANSWER HERE###

2. **Required Documentation (2 points)**  
   - For any part where you used **generative AI**, you must include the exact prompts you entered and the corresponding AI outputs. **Copy and paste them directly**.  
   - For any part where you used other external resources, list those sources.  
   - For parts completed without external resources, briefly state what prior knowledge you relied on (no detailed explanation required).

Responses that do not include **prompts and AI outputs** (when applicable) will not receive full credit.


In [8]:
### YOUR ANSWER HERE##
#YOUR PROMPTS##

##AI OUTPUTS##

(c) (5 points): Fit a ridge regression model on the training set, with λ chosen by cross-validation. Report the test error obtained.

In [9]:
###ANSWER###
# ----- Ridge with lambda chosen by CV on TRAINING set -----
alphas = np.arange(1, 2000, 1)

cv = KFold(n_splits=5, shuffle=True, random_state=42)

ridge_cv = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", RidgeCV(alphas=alphas, cv=cv, scoring="neg_root_mean_squared_error"))
])

# Fit on training only
ridge_cv.fit(X_train, y_train)

best_lambda = ridge_cv.named_steps["ridge"].alpha_

# Test predictions + RMSE
y_pred = ridge_cv.predict(X_test)
test_rmse = root_mean_squared_error(y_test, y_pred)

print("Best lambda (CV on train):", best_lambda)
print("Ridge test RMSE:", round(test_rmse, 2))

Best lambda (CV on train): 1
Ridge test RMSE: 1338.62


(d)(5 points): Fit a lasso model on the training set, with λ chosen by cross-
validation. Report the test error obtained, along with the num-
ber of non-zero coefficient estimates.

In [10]:
###ANSWER###
# Cross-validation setup
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# Lasso with cross-validation
lasso_cv = Pipeline([
    ("scaler", StandardScaler()),
    ("lasso", LassoCV(cv=cv, max_iter=10000, random_state=42))
])

# Fit on training set
lasso_cv.fit(X_train, y_train)

# Best lambda
best_lambda = lasso_cv.named_steps["lasso"].alpha_

# Test predictions
y_pred = lasso_cv.predict(X_test)

# Test RMSE
test_rmse = root_mean_squared_error(y_test, y_pred)

# Number of non-zero coefficients
coef = lasso_cv.named_steps["lasso"].coef_
num_nonzero = np.sum(coef != 0)

print("Best lambda:", best_lambda)
print("Lasso Test RMSE:", round(test_rmse, 2))
print("Number of non-zero coefficients:", num_nonzero)

Best lambda: 26.624914585759512
Lasso Test RMSE: 1325.17
Number of non-zero coefficients: 12


#### ✅ Question (d): Documenting Your Solution Process (3 points)

Please answer the following clearly and completely:

1. **Prior Knowledge vs. External Resources (1 point)**  
   Indicate which parts of Question (d) you completed using your own prior knowledge, and which parts you completed using external resources (e.g., generative AI, past assignments, Stack Overflow, Google, etc.).

In [11]:
###YOUR ANSWER HERE###

2. **Required Documentation (2 points)**  
   - For any part where you used **generative AI**, you must include the exact prompts you entered and the corresponding AI outputs. **Copy and paste them directly**.  
   - For any part where you used other external resources, list those sources.  
   - For parts completed without external resources, briefly state what prior knowledge you relied on (no detailed explanation required).

Responses that do not include **prompts and AI outputs** (when applicable) will not receive full credit.


In [12]:
### YOUR ANSWER HERE##
#YOUR PROMPTS##

##AI OUTPUTS##

(e)(5 points): Fit a PCR model on the training set, with M chosen by cross-
validation. Report the test error obtained, along with the value
of M selected by cross-validation.

In [13]:
###ANSWER###
# Cross-validation setup
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# PCR pipeline
pcr = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA()),
    ("reg", LinearRegression())
])

# Tune number of components M
param_grid = {
    "pca__n_components": np.arange(1, X_train.shape[1] + 1)
}

grid = GridSearchCV(
    pcr,
    param_grid,
    cv=cv,
    scoring="neg_root_mean_squared_error"
)

# Fit only on training data
grid.fit(X_train, y_train)

# Best M
best_M = grid.best_params_["pca__n_components"]

# Test prediction
y_pred = grid.predict(X_test)
test_rmse = root_mean_squared_error(y_test, y_pred)

print("Selected M:", best_M)
print("PCR Test RMSE:", round(test_rmse, 2))

Selected M: 17
PCR Test RMSE: 1348.33


(f) (5 points): Fit a PLS model on the training set, with M chosen by cross-
validation. Report the test error obtained, along with the value
of M selected by cross-validation.

In [14]:
###ANSWER###
# Cross-validation setup (on TRAINING set only)
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# PLS pipeline (scaling is important)
pls_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("pls", PLSRegression())
])

# Candidate number of components M:
# PLSRegression requires 1 <= n_components <= min(n_samples, n_features)
max_M = min(X_train.shape[0] - 1, X_train.shape[1])
param_grid = {"pls__n_components": np.arange(1, max_M + 1)}

grid = GridSearchCV(
    pls_pipe,
    param_grid=param_grid,
    cv=cv,
    scoring="neg_root_mean_squared_error"
)

# Fit on training set
grid.fit(X_train, y_train)

best_M = grid.best_params_["pls__n_components"]

# Predict on test set
y_pred = grid.predict(X_test).ravel()

test_rmse = root_mean_squared_error(y_test, y_pred)

print("Selected M:", best_M)
print("PLS Test RMSE:", round(test_rmse, 2))

Selected M: 11
PLS Test RMSE: 1338.68


#### ✅ Question (f): Documenting Your Solution Process (3 points)

Please answer the following clearly and completely:

1. **Prior Knowledge vs. External Resources (1 point)**  
   Indicate which parts of Question (f) you completed using your own prior knowledge, and which parts you completed using external resources (e.g., generative AI, past assignments, Stack Overflow, Google, etc.).

In [15]:
###YOUR ANSWER HERE###

2. **Required Documentation (2 points)**  
   - For any part where you used **generative AI**, you must include the exact prompts you entered and the corresponding AI outputs. **Copy and paste them directly**.  
   - For any part where you used other external resources, list those sources.  
   - For parts completed without external resources, briefly state what prior knowledge you relied on (no detailed explanation required).

Responses that do not include **prompts and AI outputs** (when applicable) will not receive full credit.


In [16]:
### YOUR ANSWER HERE##
#YOUR PROMPTS##

##AI OUTPUTS##

(g) (5 points): Comment on the results obtained. How accurately can we pre-
dict the number of college applications received? Is there much
difference among the test errors resulting from these five ap-
proaches?

###ANSWER### All five methods produce very similar test errors, indicating that no approach substantially outperforms the others. Since the number of predictors is moderate relative to the sample size, ordinary least squares already performs well, and regularization or dimension reduction only provides minor improvements. The model predicts the number of applications with moderate accuracy, but prediction errors are still sizable, especially for smaller institutions.